# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kunal452/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [7]:
# Task formulation metadata check
task_spec = {
    "Task Paradigm": "Supervised Learning",
    "Task Type": "Binary Classification with Calibrated Probability Scoring",
    "Entity Granularity": "(query_id, url_id, snapshot_date)",
    "Prediction Horizon": "30 days forward",
    "Decision Support Action": "Ranked content refresh triage queue"
}

for key, val in task_spec.items():
    print(f"{key:25}: {val}")

Task Paradigm            : Supervised Learning
Task Type                : Binary Classification with Calibrated Probability Scoring
Entity Granularity       : (query_id, url_id, snapshot_date)
Prediction Horizon       : 30 days forward
Decision Support Action  : Ranked content refresh triage queue


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [8]:
import pandas as pd
import numpy as np

def label_decay(baseline_pos, future_pos, baseline_clicks, future_clicks):
    """Computes target decay label Y in {0, 1}."""
    pos_drop = future_pos - baseline_pos
    click_ratio = future_clicks / np.maximum(baseline_clicks, 1)

    # 1 if position worsens by >= 3 AND clicks drop by >= 20%
    is_decay = (pos_drop >= 3.0) & (click_ratio <= 0.80)
    return is_decay.astype(int)

# Verification check on mock transitions
test_records = pd.DataFrame({
    'baseline_pos': [3.1, 5.0, 1.2, 8.0],
    'future_pos': [7.2, 4.9, 1.3, 12.0],
    'baseline_clicks': [1200, 450, 3100, 80],
    'future_clicks': [800, 490, 3050, 40]
})

test_records['target_decay'] = label_decay(
    test_records['baseline_pos'], test_records['future_pos'],
    test_records['baseline_clicks'], test_records['future_clicks']
)
print("Target label logic output:")
print(test_records[['baseline_pos', 'future_pos', 'baseline_clicks', 'future_clicks', 'target_decay']])

Target label logic output:
   baseline_pos  future_pos  baseline_clicks  future_clicks  target_decay
0           3.1         7.2             1200            800             1
1           5.0         4.9              450            490             0
2           1.2         1.3             3100           3050             0
3           8.0        12.0               80             40             1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [9]:
from sklearn.metrics import average_precision_score, precision_score

# Metric demonstration with simulated ground truth & predicted risk scores
y_true = np.array([0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1])
y_scores = np.array([0.10, 0.22, 0.15, 0.88, 0.30, 0.75, 0.20, 0.18, 0.92, 0.40, 0.68, 0.25, 0.12, 0.35, 0.82])

pr_auc = average_precision_score(y_true, y_scores)
top_k = 5
top_k_indices = np.argsort(y_scores)[::-1][:top_k]
precision_at_k = precision_score(y_true[top_k_indices], np.ones(top_k, dtype=int))

print(f"Observed PR-AUC: {pr_auc:.3f}")
print(f"Observed Precision@{top_k}: {precision_at_k:.2%}")

Observed PR-AUC: 1.000
Observed Precision@5: 100.00%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
# Unit of Analysis DataFrame Slice
data_slice = pd.DataFrame([
    {
        "query_id": "q_1042",
        "url_id": "url_881",
        "snapshot_date": "2026-03-01",
        "monthly_impressions": 14200,
        "avg_pos_30d": 4.2,
        "pos_std_30d": 1.85,
        "days_since_refresh": 180,
        "competitor_activity_index": 3.4,
        "target_decay_30d": 1
    },
    {
        "query_id": "q_2391",
        "url_id": "url_102",
        "snapshot_date": "2026-03-01",
        "monthly_impressions": 5800,
        "avg_pos_30d": 2.1,
        "pos_std_30d": 0.42,
        "days_since_refresh": 45,
        "competitor_activity_index": 0.2,
        "target_decay_30d": 0
    },
    {
        "query_id": "q_5890",
        "url_id": "url_415",
        "snapshot_date": "2026-03-01",
        "monthly_impressions": 31000,
        "avg_pos_30d": 8.7,
        "pos_std_30d": 2.40,
        "days_since_refresh": 320,
        "competitor_activity_index": 4.8,
        "target_decay_30d": 1
    }
])

print(f"Dataframe Shape: {data_slice.shape} (Rows: {data_slice.shape[0]}, Features: {data_slice.shape[1]})")
display(data_slice)

Dataframe Shape: (3, 9) (Rows: 3, Features: 9)


,query_id,url_id,snapshot_date,monthly_impressions,avg_pos_30d,pos_std_30d,days_since_refresh,competitor_activity_index,target_decay_30d
0,q_1042,url_881,2026-03-01,14200,4.2,1.85,180,3.4,1
1,q_2391,url_102,2026-03-01,5800,2.1,0.42,45,0.2,0
2,q_5890,url_415,2026-03-01,31000,8.7,2.40,320,4.8,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [11]:
# Demonstrating failure mode of naive heuristic rules vs. multivariate ground truth
test_scenarios = pd.DataFrame({
    'days_since_refresh': [250, 30, 200, 20],
    'competitor_activity_index': [0.1, 4.2, 3.8, 0.2],
    'true_decay_observed': [0, 1, 1, 0]
})

# Naive fixed rule
test_scenarios['naive_rule_pred'] = (test_scenarios['days_since_refresh'] > 90).astype(int)
test_scenarios['rule_correct'] = test_scenarios['naive_rule_pred'] == test_scenarios['true_decay_observed']

print("Fixed Rule Error Audit:")
print(test_scenarios[['days_since_refresh', 'competitor_activity_index', 'true_decay_observed', 'naive_rule_pred', 'rule_correct']])
print(f"\nRule Failure Rate: {(1 - test_scenarios['rule_correct'].mean()):.0%}")

Fixed Rule Error Audit:
   days_since_refresh  competitor_activity_index  true_decay_observed  \
0                 250                        0.1                    0   
1                  30                        4.2                    1   
2                 200                        3.8                    1   
3                  20                        0.2                    0   

   naive_rule_pred  rule_correct  
0                1         False  
1                0         False  
2                1          True  
3                0          True  

Rule Failure Rate: 50%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.